In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os
from glob import glob

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.figsize": (5, 4),
})


In [ ]:
# ==========================================
# 1. Helper functions
# ==========================================

base_dir = "./"

def discover_temperatures(base_dir="./"):
    summary = os.path.join(base_dir, "summary_all.csv")
    if os.path.exists(summary):
        df = pd.read_csv(summary)
        return np.sort(df['T'].dropna().unique())
    vals = []
    for path in glob(os.path.join(base_dir, "T_*")):
        try:
            vals.append(float(os.path.basename(path).split("_", 1)[1]))
        except ValueError:
            pass
    return np.array(sorted(vals))

def find_T_dir(T, base_dir="./"):
    candidates = [f"T_{T}", f"T_{T:g}", f"T_{T:.3f}", f"T_{T:.2f}"]
    for name in candidates:
        path = os.path.join(base_dir, name)
        if os.path.isdir(path):
            return path
    matches = glob(os.path.join(base_dir, f"T_{T:.3f}".rstrip('0').rstrip('.') + "*"))
    return matches[0] if matches else os.path.join(base_dir, f"T_{T:g}")

def read_dos(data_dir, filename="spectra_dos.csv"):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        print(f"Warning: File not found {file_path}")
        return None
    df = pd.read_csv(file_path)
    if df.empty:
        print(f"Warning: Empty file {file_path}")
        return None
    return df.sort_values('omega')

def get_dos_at_omega0(df, value_col='DOS', err_col='DOS_Error'):
    omega = df['omega'].to_numpy()
    vals = df[value_col].to_numpy()
    idx0 = int(np.argmin(np.abs(omega)))
    err0 = np.nan
    if err_col in df.columns:
        err0 = df[err_col].to_numpy()[idx0]
    return vals[idx0], err0, float(omega[idx0])

def estimate_tc_from_summary(base_dir="./", rho_col='Superfluid_Stiffness_mean'):
    summary = os.path.join(base_dir, "summary_all.csv")
    if not os.path.exists(summary):
        return np.nan
    df = pd.read_csv(summary).sort_values('T')
    if rho_col not in df.columns or len(df) < 2:
        return np.nan
    T = df['T'].to_numpy(dtype=float)
    diff = df[rho_col].to_numpy(dtype=float) - (2.0 / np.pi) * T
    finite = np.isfinite(T) & np.isfinite(diff)
    T = T[finite]
    diff = diff[finite]
    if len(T) < 2:
        return np.nan
    for i in range(len(T) - 1):
        if diff[i] == 0:
            return float(T[i])
        if diff[i] * diff[i + 1] <= 0:
            return float(T[i] - diff[i] * (T[i + 1] - T[i]) / (diff[i + 1] - diff[i]))
    return np.nan

def temperature_norm(T_values):
    T_values = np.asarray(T_values, dtype=float)
    cmap = plt.cm.turbo
    if len(T_values) > 1 and np.nanmax(T_values) > np.nanmin(T_values):
        norm = mcolors.Normalize(vmin=np.nanmin(T_values), vmax=np.nanmax(T_values))
    else:
        t0 = float(T_values[0]) if len(T_values) else 0.0
        norm = mcolors.Normalize(vmin=t0 - 1e-12, vmax=t0 + 1e-12)
    return cmap, norm

def nearest_temperature(T_values, target):
    T_values = np.asarray(T_values, dtype=float)
    if len(T_values) == 0 or not np.isfinite(target):
        return np.nan
    return float(T_values[np.nanargmin(np.abs(T_values - target))])


In [ ]:
# ==========================================
# 2. Main loop
# ==========================================

T_list = discover_temperatures(base_dir)
T_list = T_list[(T_list >= 0.005) & (T_list <= 0.100)]

results = {"T": [], "N0": [], "N0_err": []}
spectra = []
Tc = estimate_tc_from_summary(base_dir)
T_near_Tc = nearest_temperature(T_list, Tc)

print(f"Starting DOS analysis for {len(T_list)} temperatures...")

for T in T_list:
    data_dir = find_T_dir(T, base_dir)
    print(f"Processing {os.path.basename(data_dir)}...", end='\r')
    df = read_dos(data_dir)
    if df is None:
        continue

    omega = df['omega'].to_numpy()
    dos = df['DOS'].to_numpy()
    dos_err = df['DOS_Error'].to_numpy() if 'DOS_Error' in df.columns else None
    spectra.append({'T': T, 'omega': omega, 'dos': dos, 'dos_err': dos_err})

    n0, n0_err, omega0 = get_dos_at_omega0(df, 'DOS', 'DOS_Error')
    results['T'].append(T)
    results['N0'].append(n0)
    results['N0_err'].append(n0_err)

print()
print("Analysis complete.")
if np.isfinite(Tc):
    print(f"Tc from Kubo-BKT crossing: {Tc:.5f}; highlighted T={T_near_Tc:g}")
for key in results:
    results[key] = np.array(results[key])


In [ ]:
# ==========================================
# 3. N(omega) vs omega for different T
# ==========================================

if len(spectra) == 0:
    raise RuntimeError("No valid DOS data found.")

fig, ax = plt.subplots(dpi=300)
T_used = np.array([spec['T'] for spec in spectra], dtype=float)
cmap, norm = temperature_norm(T_used)

for spec in spectra:
    highlight = np.isfinite(T_near_Tc) and np.isclose(spec['T'], T_near_Tc)
    color = 'black' if highlight else cmap(norm(spec['T']))
    linewidth = 3.0 if highlight else 1.3
    zorder = 5 if highlight else 2
    label = rf"T={spec['T']:g} closest to $T_c$" if highlight else None
    ax.plot(spec['omega'], spec['dos'], color=color, linewidth=linewidth,
            zorder=zorder, label=label)

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'$T$')

ax.set_xlabel(r'$\omega$')
ax.set_ylabel(r'$N(\omega)$')
if np.isfinite(T_near_Tc):
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 4. N(omega=0) vs T
# ==========================================

fig, ax = plt.subplots(dpi=300)
mask = np.isfinite(results['T']) & np.isfinite(results['N0'])
ax.errorbar(results['T'][mask], results['N0'][mask], yerr=results['N0_err'][mask],
            fmt='-o', color='black', capsize=3, label=r'$N(0)$')
if np.isfinite(Tc):
    ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5,
               label=rf'$T_c={Tc:.4f}$')
ax.set_xlabel(r'$T$')
ax.set_ylabel(r'$N(0)$')
ax.set_xlim(0.0, 0.105)
ax.legend(frameon=False)
plt.show()


In [ ]:
# ==========================================
# 5. dN(omega=0)/dT vs T
# ==========================================

t_vals = results['T'].copy()
n0_vals = results['N0'].copy()

mask = np.isfinite(t_vals) & np.isfinite(n0_vals)
t_vals = t_vals[mask]
n0_vals = n0_vals[mask]

sort_idx = np.argsort(t_vals)
t_vals = t_vals[sort_idx]
n0_vals = n0_vals[sort_idx]

if len(t_vals) < 2:
    print("Not enough points to compute dN/dT.")
else:
    dN_dT = np.gradient(n0_vals, t_vals)

    fig, ax = plt.subplots(dpi=300)
    ax.plot(t_vals, dN_dT, '-o', color='blue')
    ax.set_xlabel(r'$T$')
    ax.set_ylabel(r'$dN(\omega=0)/dT$')
    plt.show()
